# TPC Ablation Study - Google Colab

**OPTION 1 (RECOMMENDED): Upload your local data to Google Drive**
- On your PC: Upload `C:\cs598\mimic-iv` folder to Google Drive root
- Skip cell 4 below
- In cell 5, set `MIMIC_ROOT = '/content/drive/MyDrive/mimic-iv'`

**OPTION 2: Download from shared link (slower, may fail)**
- Run cell 4 and wait for "✓ READY!" message
- If it fails, use Option 1 instead

**Steps:**
1. Reset runtime if rerunning: Runtime → Disconnect and delete runtime
2. Enable GPU: Runtime → Change runtime type → T4 GPU → Save
3. Run cells 1-3 (clone, install, mount Drive)
4. Choose option above (upload OR download)
5. Run cell 5 to configure paths (verify no errors)
6. Run cell 6 to train (2-4 hours)

In [ ]:
# Clone repository (only if not already cloned)
import os
if not os.path.exists('/content/PyHealth'):
    !git clone https://github.com/tarakjc2c/PyHealth.git
    print('✓ Cloned repository')
else:
    print('✓ Repository already exists')

%cd /content/PyHealth
!git checkout pr-1028

In [ ]:
# Install dependencies
!pip install -e . -q
!pip install litdata polars pandas dask mne rdkit peft transformers ogb -q
import os
print(f'✓ Installed in: {os.getcwd()}')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# OPTION 2: Download MIMIC-IV from shared Google Drive
# If you uploaded to your own Drive (Option 1), skip this cell!

!pip install gdown -q
import os
import shutil

MIMIC_ROOT = '/content/mimic-iv'
GDRIVE_FOLDER_ID = '15vyfKQ6H0g7DVEbI8vAMNhr4fi2lznVn'

# Check if data already exists
if os.path.exists(f'{MIMIC_ROOT}/hosp/diagnoses_icd.csv.gz'):
    print('✓ MIMIC-IV data already exists')
    hosp_files = len([f for f in os.listdir(f'{MIMIC_ROOT}/hosp') if f.endswith('.csv.gz')])
    icu_files = len([f for f in os.listdir(f'{MIMIC_ROOT}/icu') if f.endswith('.csv.gz')])
    print(f'✓ Found {hosp_files} hosp files, {icu_files} icu files')
elif os.path.exists('/content/drive/MyDrive/mimic-iv/hosp/diagnoses_icd.csv.gz'):
    print('✓ Found data in your Google Drive!')
    print('You can skip this cell. In cell 5, use:')
    print('MIMIC_ROOT = "/content/drive/MyDrive/mimic-iv"')
else:
    print('Downloading MIMIC-IV data (9.92 GB, ~15-20 minutes)...')
    print('⚠ This may fail. If so, upload C:\\cs598\\mimic-iv to your Drive instead.\n')
    
    # Download to temporary location
    temp_dir = '/content/mimic-download'
    !gdown --folder {GDRIVE_FOLDER_ID} -O {temp_dir} --remaining-ok
    
    print('\nOrganizing files...')
    # Find where gdown put the files
    if os.path.exists(temp_dir):
        !ls -la {temp_dir}
        
        # Check for nested folder structure
        contents = os.listdir(temp_dir)
        print(f'Downloaded contents: {contents}')
        
        # Look for hosp and icu folders
        found = False
        for item in contents:
            item_path = os.path.join(temp_dir, item)
            if os.path.isdir(item_path):
                # Check if this folder contains hosp/icu
                if os.path.exists(f'{item_path}/hosp') and os.path.exists(f'{item_path}/icu'):
                    shutil.move(item_path, MIMIC_ROOT)
                    found = True
                    break
        
        # If hosp/icu are directly in temp_dir
        if not found and 'hosp' in contents and 'icu' in contents:
            shutil.move(temp_dir, MIMIC_ROOT)
            found = True
        
        if found:
            print(f'✓ Data organized at {MIMIC_ROOT}')
        else:
            print('✗ ERROR: Could not find hosp/icu folders in download')
            print('Please upload C:\\cs598\\mimic-iv to your Google Drive manually')
    else:
        print('✗ ERROR: Download failed')
        print('Please upload C:\\cs598\\mimic-iv to your Google Drive manually')

# Final verification
print('\n=== VERIFICATION ===')
locations = ['/content/mimic-iv', '/content/drive/MyDrive/mimic-iv']
for loc in locations:
    if os.path.exists(f'{loc}/hosp/diagnoses_icd.csv.gz'):
        print(f'✓ READY! Data found at: {loc}')
        hosp_files = len([f for f in os.listdir(f'{loc}/hosp') if f.endswith('.csv.gz')])
        icu_files = len([f for f in os.listdir(f'{loc}/icu') if f.endswith('.csv.gz')])
        print(f'  {hosp_files} hosp files, {icu_files} icu files')
        print(f'\n➜ In cell 5, use: MIMIC_ROOT = "{loc}"')
        break
else:
    print('✗ Data not found. Upload C:\\cs598\\mimic-iv to Google Drive, then rerun this cell.')

In [ ]:
# Configure paths - UPDATE MIMIC_ROOT if cell 4 said to
import os

# Set based on where your data is (cell 4 told you which to use)
MIMIC_ROOT = '/content/mimic-iv'  # OR '/content/drive/MyDrive/mimic-iv' if uploaded to Drive

# Verify data exists BEFORE configuring script
if not os.path.exists(f'{MIMIC_ROOT}/hosp/diagnoses_icd.csv.gz'):
    print(f'✗ ERROR: Data not found at {MIMIC_ROOT}')
    print('\nSearching for data...')
    !find /content -name "diagnoses_icd.csv.gz" 2>/dev/null | head -5
    print('\n⚠ UPDATE MIMIC_ROOT variable above to the correct path, then rerun this cell')
    raise RuntimeError('Data not found - check MIMIC_ROOT path')

print(f'✓ Data verified at: {MIMIC_ROOT}')

# Configure script paths
script = 'examples/length_of_stay/length_of_stay_mimic4_tpc.py'

with open(script, 'r') as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if 'MIMIC_ROOT = r"C:' in line:
        new_lines.append(f'MIMIC_ROOT = "{MIMIC_ROOT}"\n')
    elif 'CACHE_PATH = r"C:' in line:
        new_lines.append('CACHE_PATH = "/content/tpc_cache"\n')
    elif 'OUTPUT_DIR = "tpc_ablation_results"' in line:
        new_lines.append('OUTPUT_DIR = "/content/drive/MyDrive/tpc_ablation_results"\n')
    else:
        new_lines.append(line)

with open(script, 'w') as f:
    f.writelines(new_lines)

print(f'✓ Script configured:')
print(f'  MIMIC_ROOT = {MIMIC_ROOT}')
print(f'  CACHE_PATH = /content/tpc_cache')
print(f'  OUTPUT_DIR = /content/drive/MyDrive/tpc_ablation_results')

In [ ]:
# Run ablation study (2-4 hours)
!python examples/length_of_stay/length_of_stay_mimic4_tpc.py

## Download Results

Results are in MyDrive/tpc_ablation_results/
Download them with the cell below:

In [ ]:
from google.colab import files
import os

result_dir = '/content/drive/MyDrive/tpc_ablation_results'
for filename in ['ablation_results.json', 'mc_dropout_results.json']:
    filepath = os.path.join(result_dir, filename)
    if os.path.exists(filepath):
        print(f'Downloading: {filename}')
        files.download(filepath)